<a href="https://colab.research.google.com/github/Bernpro/TaskFlow-Python-Data-Automation-3/blob/main/Task_Flow_2_modules.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import openpyxl
import pandas as pd
from bs4 import BeautifulSoup
from google.colab import files  # <-- LIBRERÍA PARA SUBIR ARCHIVOS EN COLAB
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

# =====================================================================
# INTERFAZ: CARGA DE ARCHIVO EXCEL DESDE TU COMPUTADORA
# =====================================================================
def cargar_archivo_usuario():
    print("📁 [PASO 1]: Por favor, sube tu archivo Excel crudo...")

    # Esto abre el selector de archivos nativo de tu computadora en Colab
    archivos_subidos = files.upload()

    if not archivos_subidos:
        print("❌ No se subió ningún archivo. Proceso cancelado.")
        return None

    # Obtener el nombre del archivo que el usuario seleccionó
    nombre_archivo = list(archivos_subidos.keys())[0]
    print(f"✅ Archivo '{nombre_archivo}' cargado exitosamente en la nube.")
    return nombre_archivo


# =====================================================================
# MÓDULO 2: AUTOMATIZACIÓN DE REPORTES (Excel Corporativo)
# =====================================================================
def generar_reporte_automatizado(archivo_entrada):
    print("\n📊 [PASO 2]: Leyendo datos y transformando en reporte ejecutivo...")

    if not archivo_entrada or not os.path.exists(archivo_entrada):
        print("❌ Error: El archivo de origen no está disponible.")
        return None

    try:
        # 1. Leer el archivo cargado (Soporta .xlsx y .xls)
        df_datos = pd.read_excel(archivo_entrada)

        # Validar si el Excel tiene las columnas correctas
        if "Producto" not in df_datos.columns or "Precio_Base" not in df_datos.columns:
            print("⚠️ Advertencia: El archivo debe contener las columnas 'Producto' y 'Precio_Base'.")
            print("Generando datos de simulación estructurados para evitar fallos...")
            datos_simulados = [
                {"Producto": "Laptop Premium X15", "Precio_Base": 24500},
                {"Producto": "Monitor Gamer 4K 27''", "Precio_Base": 8900},
                {"Producto": "Teclado Mecánico RGB", "Precio_Base": 1850},
                {"Producto": "Mouse Ergonómico Inalámbrico", "Precio_Base": 1200}
            ]
            df_datos = pd.DataFrame(datos_simulados)

        # Procesamiento Lógico y Fórmulas Financieras
        df_datos["Precio_Base"] = pd.to_numeric(df_datos["Precio_Base"])
        df_datos["IVA (16%)"] = df_datos["Precio_Base"] * 0.16
        df_datos["Precio_Final"] = df_datos["Precio_Base"] + df_datos["IVA (16%)"]

        nombre_salida = "Reporte_Automatizado_Final.xlsx"

        # Guardar la estructura base con Pandas
        with pd.ExcelWriter(nombre_salida, engine="openpyxl") as writer:
            df_datos.to_excel(writer, index=False, sheet_name="Monitoreo")

        # 2. Inyección de Diseño Ejecutivo con OpenPyXL
        wb = openpyxl.load_workbook(nombre_salida)
        ws = wb["Monitoreo"]

        # Estilos visuales
        azul_oscuro = PatternFill(start_color="1F4E78", end_color="1F4E78", fill_type="solid")
        azul_claro_filas = PatternFill(start_color="F2F5F8", end_color="F2F5F8", fill_type="solid")
        fuente_encabezado = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
        fuente_datos = Font(name="Calibri", size=11, bold=False)
        fuente_total = Font(name="Calibri", size=11, bold=True)

        borde_delgado = Side(border_style="thin", color="D9D9D9")
        borde_doble = Side(border_style="double", color="000000")
        cuadro_datos = Border(left=borde_delgado, right=borde_delgado, top=borde_delgado, bottom=borde_delgado)
        borde_subtotal = Border(top=borde_delgado, bottom=borde_doble)

        # Aplicar estilos al encabezado (Fila 1)
        ws.row_dimensions[1].height = 26
        for col in range(1, ws.max_column + 1):
            celda = ws.cell(row=1, column=col)
            celda.fill = azul_oscuro
            celda.font = fuente_encabezado
            celda.alignment = Alignment(horizontal="center", vertical="center")

        # Formatear filas de datos (Estilo Cebra)
        for fila in range(2, ws.max_row + 1):
            ws.row_dimensions[fila].height = 20

            # Columna Producto
            celda_prod = ws.cell(row=fila, column=1)
            celda_prod.alignment = Alignment(horizontal="left", vertical="center")
            celda_prod.font = fuente_datos
            celda_prod.border = cuadro_datos

            # Columnas numéricas (Precio_Base, IVA, Precio_Final)
            for col in range(2, 5):
                celda_num = ws.cell(row=fila, column=col)
                celda_num.number_format = "$#,##0.00"
                celda_num.alignment = Alignment(horizontal="right", vertical="center")
                celda_num.font = fuente_datos
                celda_num.border = cuadro_datos

                if fila % 2 == 0:
                    celda_num.fill = azul_claro_filas
                    celda_prod.fill = azul_claro_filas

        # 3. Insertar fila automática de promedios con fórmulas nativas
        fila_total = ws.max_row + 1
        ws.row_dimensions[fila_total].height = 22
        ws.cell(row=fila_total, column=1, value="PROMEDIO GENERAL").font = fuente_total

        for col, letra in zip([2, 3, 4], ["B", "C", "D"]):
            celda_formula = ws.cell(row=fila_total, column=col)
            celda_formula.value = f"=AVERAGE({letra}2:{letra}{fila_total-1})"
            celda_formula.font = fuente_total
            celda_formula.number_format = "$#,##0.00"
            celda_formula.alignment = Alignment(horizontal="right")
            celda_formula.border = borde_subtotal

        # Ajustar ancho de columnas dinámicamente
        for col in ws.columns:
            max_largo = max(len(str(cell.value or "")) for cell in col)
            ws.column_dimensions[col.column_letter].width = max(max_largo + 4, 15)

        wb.save(nombre_salida)
        print(f"✅ Reporte diseñado con éxito corporativo: '{nombre_salida}'")
        return nombre_salida

    except Exception as e:
        print(f"❌ Error al procesar el archivo Excel: {e}")
        return None


# =====================================================================
# CONFIGURACIÓN DE ARRANQUE
# =====================================================================
if __name__ == "__main__":
    print("🚀 PROCESADOR AUTOMÁTICO DE EXCEL INICIADO 🚀\n")

    # 1. El usuario sube el archivo
    archivo_usuario = cargar_archivo_usuario()

    # 2. Si el archivo es válido, se procesa el diseño automáticamente
    if archivo_usuario:
        generar_reporte_automatizado(archivo_usuario)
        print("\n🎉 ¡Automatización completada! Descarga 'Reporte_Automatizado_Final.xlsx' de la barra lateral.")


🚀 PROCESADOR AUTOMÁTICO DE EXCEL INICIADO 🚀

📁 [PASO 1]: Por favor, sube tu archivo Excel crudo...


❌ No se subió ningún archivo. Proceso cancelado.
